# COMP315 TFMA Analysis Notebook

This is the official final TFMA notebook for the project.

Required sections included:
- overall TFMA results
- slicing by sex
- slicing by race
- fairness indicator
- counterfactual analysis
- fairness threshold analysis


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_model_analysis as tfma
import tensorflow_transform as tft

# find repo root from current working directory

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'tfx_pipeline_output_v2').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('could not find repo root from current working directory')

PROJECT_ROOT = find_repo_root(Path.cwd().resolve())
print(f'project root: {PROJECT_ROOT}')


2026-04-15 23:41:47.148048: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-15 23:41:47.166060: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-15 23:41:47.261206: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-15 23:41:47.358763: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-15 23:41:47.437284: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

project root: /mnt/c/Users/said_/Documents/COMP-315-Team-Project


In [2]:
# load latest evaluator output
EVAL_BASE_DIR = PROJECT_ROOT / 'tfx_pipeline_output_v2' / 'Evaluator' / 'evaluation'
if not EVAL_BASE_DIR.exists():
    raise FileNotFoundError(f'missing evaluator directory: {EVAL_BASE_DIR}')

run_dirs = sorted(
    [d for d in EVAL_BASE_DIR.iterdir() if d.is_dir() and d.name.isdigit()],
    key=lambda d: int(d.name),
)
if not run_dirs:
    raise FileNotFoundError(f'no eval runs found in: {EVAL_BASE_DIR}')

EVAL_DIR = run_dirs[-1]
eval_result = tfma.load_eval_result(output_path=str(EVAL_DIR))
print(f'loaded eval from: {EVAL_DIR}')


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


loaded eval from: /mnt/c/Users/said_/Documents/COMP-315-Team-Project/tfx_pipeline_output_v2/Evaluator/evaluation/8


In [3]:
tfma.view.render_slicing_metrics(eval_result)

SlicingMetricsViewer(config={'weightedExamplesColumn': 'example_count'}, data=[{'slice': 'Overall', 'metrics':…

### Interpretation: Overall TFMA Results

The overall metrics show the model has solid baseline classification quality on the evaluation split. This is useful for a global summary, but overall values alone can hide subgroup differences, so slice-based checks are still necessary.


In [4]:
tfma.view.render_slicing_metrics(eval_result, slicing_column='sex')

SlicingMetricsViewer(config={'weightedExamplesColumn': 'example_count'}, data=[{'slice': 'sex:Male', 'metrics'…

### Interpretation: Slicing by Sex

The sex slices show a measurable performance gap between groups. This means the model does not perform equally for this sensitive feature, and fairness should be monitored with subgroup metrics instead of only global averages.


In [5]:
tfma.view.render_slicing_metrics(eval_result, slicing_column='race')

SlicingMetricsViewer(config={'weightedExamplesColumn': 'example_count'}, data=[{'slice': 'race:White', 'metric…

### Interpretation: Slicing by Race

The race slices also show uneven behavior across groups. This confirms that aggregate accuracy can hide subgroup-level issues and supports using slice diagnostics as a standard part of evaluation.


In [6]:
# render fairness indicator
# handle both older and newer tfma api layouts
if hasattr(tfma.view, 'render_fairness_indicator'):
    try:
        tfma.view.render_fairness_indicator(eval_result, slicing_column='sex')
    except TypeError:
        tfma.view.render_fairness_indicator(eval_result)
else:
    try:
        from tensorflow_model_analysis.addons.fairness.view import widget_view

        try:
            widget_view.render_fairness_indicator(eval_result=eval_result, slicing_column='sex')
        except TypeError:
            widget_view.render_fairness_indicator(eval_result=eval_result)
    except Exception as exc:
        # keep run all stable if fairness widget is unavailable
        print(f'fairness widget not available in this environment: {exc}')
        tfma.view.render_slicing_metrics(eval_result, slicing_column='sex')


fairness widget not available in this environment: No module named 'tensorflow_model_analysis.addons'


### Interpretation: Fairness Indicator

The fairness indicator gives a direct comparison of subgroup outcomes at selected thresholds. It supports the slice findings by highlighting where parity gaps remain and where threshold choices can change the observed fairness pattern.


In [7]:
# load model, transform graph, and source data
DATA_PATH = PROJECT_ROOT / 'data' / 'Dataset1_adult' / 'source' / 'adult.csv'
MODEL_ROOT = PROJECT_ROOT / 'tfx_pipeline_output_v2' / 'Trainer' / 'model'
TRANSFORM_ROOT = PROJECT_ROOT / 'tfx_pipeline_output_v2' / 'Transform' / 'transform_graph'

model_runs = sorted(
    [d for d in MODEL_ROOT.iterdir() if d.is_dir() and d.name.isdigit()],
    key=lambda d: int(d.name),
)
if not model_runs:
    raise FileNotFoundError(f'no model runs found in: {MODEL_ROOT}')

latest_model_run = model_runs[-1]
saved_model_pb = next(latest_model_run.rglob('saved_model.pb'), None)
if saved_model_pb is None:
    raise FileNotFoundError(f'no saved_model.pb found under: {latest_model_run}')

MODEL_DIR = saved_model_pb.parent
loaded_model = tf.saved_model.load(str(MODEL_DIR))
serve_fn = loaded_model.signatures['serving_default']
output_key = 'outputs' if 'outputs' in serve_fn.structured_outputs else next(iter(serve_fn.structured_outputs))

transform_runs = sorted(
    [d for d in TRANSFORM_ROOT.iterdir() if d.is_dir() and d.name.isdigit()],
    key=lambda d: int(d.name),
)
if not transform_runs:
    raise FileNotFoundError(f'no transform runs found in: {TRANSFORM_ROOT}')

TRANSFORM_DIR = transform_runs[-1]
tf_transform_output = tft.TFTransformOutput(str(TRANSFORM_DIR))
raw_feature_spec = tf_transform_output.raw_feature_spec().copy()
transformed_feature_spec = tf_transform_output.transformed_feature_spec().copy()
LABEL_KEY = 'target'
transformed_feature_spec.pop(LABEL_KEY, None)
tft_layer = tf_transform_output.transform_features_layer()

df = pd.read_csv(DATA_PATH).copy()

print(f'model dir: {MODEL_DIR}')
print(f'transform dir: {TRANSFORM_DIR}')
print(f'data rows: {len(df)}')


I0000 00:00:1776310946.395289   83269 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-15 23:42:26.404599: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2343] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


model dir: /mnt/c/Users/said_/Documents/COMP-315-Team-Project/tfx_pipeline_output_v2/Trainer/model/7/Format-Serving
transform dir: /mnt/c/Users/said_/Documents/COMP-315-Team-Project/tfx_pipeline_output_v2/Transform/transform_graph/6
data rows: 32561


In [ ]:
# build transformed tf.Example objects so serving signature input matches

def _coerce_value(value, dtype):
    if pd.isna(value):
        if dtype == tf.string:
            return ''
        if dtype == tf.int64:
            return 0
        return 0.0

    if dtype == tf.string:
        return str(value)
    if dtype == tf.int64:
        return int(value)
    return float(value)


def _build_dense_value(value, dtype, feature_shape):
    # build values that match fixedlenfeature shape
    if not feature_shape:
        if dtype == tf.string:
            return [str(value).encode('utf-8')]
        return [value]

    size = int(np.prod(feature_shape))
    dense_values = [value] * max(size, 1)
    dense_values = np.array(dense_values, dtype=object).reshape((1, *feature_shape))

    if dtype == tf.string:
        return np.vectorize(lambda x: str(x).encode('utf-8'))(dense_values)
    return dense_values


def _to_input_tensor(value, feature_spec):
    dtype = feature_spec.dtype
    value = _coerce_value(value, dtype)

    if isinstance(feature_spec, tf.io.VarLenFeature):
        if dtype == tf.string:
            values = tf.constant([str(value).encode('utf-8')], dtype=tf.string)
        elif dtype == tf.int64:
            values = tf.constant([int(value)], dtype=tf.int64)
        else:
            values = tf.constant([float(value)], dtype=tf.float32)
        return tf.SparseTensor(indices=[[0, 0]], values=values, dense_shape=[1, 1])

    # fixedlenfeature path
    feature_shape = tuple(feature_spec.shape)
    dense_value = _build_dense_value(value, dtype, feature_shape)

    if dtype == tf.string:
        return tf.constant(dense_value, dtype=tf.string)
    if dtype == tf.int64:
        return tf.constant(dense_value, dtype=tf.int64)
    return tf.constant(dense_value, dtype=tf.float32)


def row_to_transformed_example(row: pd.Series) -> tf.train.Example:
    raw_inputs = {}
    for feature_name, feature_spec in raw_feature_spec.items():
        raw_inputs[feature_name] = _to_input_tensor(row.get(feature_name), feature_spec)

    transformed = tft_layer(raw_inputs)
    ex = tf.train.Example()

    for feature_name, tensor in transformed.items():
        if feature_name == LABEL_KEY:
            continue

        values = np.array(tensor.numpy()[0]).reshape(-1)
        if values.size == 0:
            continue

        # cast using transformed spec so parse_example types always match
        expected_spec = transformed_feature_spec.get(feature_name)
        expected_dtype = expected_spec.dtype if expected_spec is not None else tensor.dtype

        if expected_dtype == tf.string:
            ex.features.feature[feature_name].bytes_list.value.extend([
                v if isinstance(v, bytes) else str(v).encode('utf-8') for v in values
            ])
        elif expected_dtype == tf.int64:
            ex.features.feature[feature_name].int64_list.value.extend([
                int(np.rint(float(v))) for v in values
            ])
        else:
            ex.features.feature[feature_name].float_list.value.extend([
                float(v) for v in values
            ])

    return ex


def custom_predict_fn(rows: pd.DataFrame):
    examples = [row_to_transformed_example(row) for _, row in rows.iterrows()]
    serialized = tf.constant([ex.SerializeToString() for ex in examples], dtype=tf.string)
    preds = serve_fn(examples=serialized)[output_key].numpy().reshape(-1)
    return np.array(preds, dtype=float)

# quick sanity check
sanity_df = df.head(3).copy()
sanity_df['p_>50k'] = custom_predict_fn(sanity_df)
sanity_df[['age', 'sex', 'race', 'hours-per-week', 'education-num', 'p_>50k']]


In [ ]:
# baseline predictions for a small sample
sample_df = df.head(5).copy()
sample_df['p_>50k'] = custom_predict_fn(sample_df)
sample_df[['age', 'sex', 'race', 'hours-per-week', 'education-num', 'p_>50k']]


### Interpretation: Counterfactual Baseline

This table shows baseline predicted probabilities on original rows before any manual edits. It gives a reference point so each counterfactual change can be compared clearly against the same starting example.


In [ ]:
# counterfactual edits on one example
base = df.iloc[0].copy()
sex_values = list(df['sex'].dropna().astype(str).str.strip().unique())
current_sex = str(base['sex']).strip()
alt_sex = next((v for v in sex_values if v != current_sex), current_sex)

cf_sex = base.copy()
cf_sex['sex'] = alt_sex

cf_hours = base.copy()
cf_hours['hours-per-week'] = 60

cf_education = base.copy()
cf_education['education-num'] = 16

compare_df = pd.DataFrame(
    [base, cf_sex, cf_hours, cf_education],
    index=['original', 'sex_changed', 'hours_60', 'education_16'],
)
compare_df['p_>50k'] = custom_predict_fn(compare_df)
compare_df[['sex', 'race', 'hours-per-week', 'education-num', 'p_>50k']]


### Interpretation: Counterfactual Analysis

The counterfactual comparisons show that changing one feature at a time can shift prediction probability in meaningful ways. Work-related features change outputs as expected, and demographic changes also affect scores, which is important for fairness review.


In [ ]:
# fairness threshold analysis by sex
analysis_df = df.head(200).copy()
analysis_df['p_>50k'] = custom_predict_fn(analysis_df)
analysis_df['sex_clean'] = analysis_df['sex'].astype(str).str.strip()
analysis_df['label'] = analysis_df['target'].astype(str).str.strip().str.contains('>50K').astype(int)


def compute_binary_metrics(y_true, y_prob, threshold):
    y_true = np.array(y_true, dtype=int)
    y_pred = (np.array(y_prob) >= threshold).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    total = max(len(y_true), 1)
    return {
        'accuracy': (tp + tn) / total,
        'tpr': tp / max(tp + fn, 1),
        'fpr': fp / max(fp + tn, 1),
    }

thresholds = np.arange(0.10, 0.95, 0.05)
sex_groups = sorted(analysis_df['sex_clean'].dropna().unique())
rows = []

for thr in thresholds:
    row = {'threshold': round(float(thr), 2)}

    for group in sex_groups:
        group_df = analysis_df[analysis_df['sex_clean'] == group]
        metrics = compute_binary_metrics(group_df['label'], group_df['p_>50k'], thr)
        row[f'{group}_accuracy'] = metrics['accuracy']
        row[f'{group}_fpr'] = metrics['fpr']
        row[f'{group}_tpr'] = metrics['tpr']

    if len(sex_groups) >= 2:
        g1, g2 = sex_groups[0], sex_groups[1]
        row['accuracy_gap'] = abs(row[f'{g1}_accuracy'] - row[f'{g2}_accuracy'])

    rows.append(row)

sex_threshold_df = pd.DataFrame(rows)
sex_threshold_df


### Interpretation: Fairness Threshold Table

The threshold table shows how subgroup metrics change as the decision cutoff changes. This helps explain that fairness outcomes are threshold-dependent and should not be judged from one cutoff alone.


In [ ]:
# plot threshold trends
plt.figure(figsize=(10, 6))

for group in sex_groups:
    plt.plot(
        sex_threshold_df['threshold'],
        sex_threshold_df[f'{group}_accuracy'],
        marker='o',
        label=f'{group} accuracy',
    )

if 'accuracy_gap' in sex_threshold_df.columns:
    plt.plot(
        sex_threshold_df['threshold'],
        sex_threshold_df['accuracy_gap'],
        marker='o',
        linestyle='--',
        label='accuracy gap',
    )

plt.xlabel('threshold')
plt.ylabel('score')
plt.title('fairness threshold analysis by sex')
plt.grid(True)
plt.legend()
plt.show()


### Interpretation: Fairness Threshold Plot

The plot shows that group curves are not perfectly aligned across thresholds. This indicates that threshold choice can increase or reduce disparity, so threshold tuning should consider both performance and fairness.
